<a href="https://colab.research.google.com/github/gowhag/CM4044-Group-14-Project-2/blob/main/%5BESOL%5D_CM4044_Prj2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CM4044 Group 14 - Project 2**
***MP2-01  One Molecule, Many Faces: Does the Representation Matter?***

**Research Question:** For the same property-prediction task, how much of the performance comes from the representation rather than the model?

**Group Members:**
*   Wan Jeng Hor (U2440397D)
*   Yap Kang Lin (U2440387F)
*   Serene Ann Bravin (U2440043D)

---
# Notebook Structure
| Phase | Content |
|---|---|
| **Deliverable 1** | Dataset Curation and Preprocessing |
| **Deliverable 2** | Molecular Representation and Model Architecture |
| **Deliverable 3** | Comparative Model Evaluation |
| **Deliverable 4** | Molecular-Level Error Analysis |



## Baseline VS. Model
Three-way (plus stretch) comparison of molecular representations on the SAME
curated dataset, SAME scaffold splits, SAME seeds:

    1. RDKit 2D descriptors  -> XGBoost            (baseline)
    2. SMILES characters     -> BiLSTM             (main model)
    3. Molecular graph       -> GCN (PyG)          (main model)
  stretch:
    4. SELFIES tokens        -> BiLSTM
    5. Mol2Vec embedding     -> XGBoost
    6. Ensembles (mean / inverse-val-MSE weighted)

Colab setup (run once in a cell):
    !pip -q install rdkit xgboost torch_geometric selfies gensim scipy scikit-learn matplotlib pandas

Usage:
    python mp2_01_representation_study.py --primary esol --confirm lipo --seeds 5
    python mp2_01_representation_study.py --primary esol --no_confirm --no_stretch --seeds 2 --epochs 20   # smoke test

Outputs (in --out, default ./results):
    <ds>_metrics_per_seed.csv, <ds>_summary.csv, <ds>_paired_tests.csv,
    <ds>_predictions.csv, <ds>_error_correlations.csv, <ds>_error_by_bin.csv,
    <ds>_model_error_correlation.csv, <ds>_hardest_molecules.csv, figures *.png

## Library Imports
All required libraries are imported here. Key packages:
- `rdkit` — parsing SMILES, molecular descriptors, Morgan fingerprints, Murcko scaffolds and standardisation
- `selfies` / `gensim` — SELFIES string encoding and Word2Vec embeddings
- `torch` — neural networks, data loaders and padded sequences for RNN-style models
- `torch_geometric` — graph neural networks (GCN layers, graph pooling and loaders)
- `xgboost` — gradient-boosted regression baseline
- `sklearn` — imputation and evaluation metrics (MAE, RMSE, R²)
- `scipy.stats` — Spearman correlation and t-distribution (significance, confidence intervals)
- `numpy` / `pandas` — arrays and tabular data
- `matplotlib` — plotting (uses the `Agg` backend, so figures are saved to file, not shown inline)

In [ ]:
#!/usr/bin/env python
!pip install -q rdkit xgboost torch_geometric selfies gensim

#
import argparse
import os
import random
import urllib.request
import warnings
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy import stats
import selfies as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

#RDKIt
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator, Lipinski
from rdkit.Chem import AllChem
from rdkit.Chem import rdMolDescriptors, Crippen
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

from xgboost import XGBRegressor
from torch.utils.data import Dataset,DataLoader
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GraphLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error
from sklearn.metrics import r2_score

from scipy.stats import t
from copy import deepcopy
from scipy.stats import spearmanr
from pathlib import Path
from gensim.models import Word2Vec

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"RDKit version: {rdkit.__version__}")
print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 24.8 MB/s eta 0:00:00
RDKit version: 2026.09.1
Setup complete!


# Dataset Curation and Preprocessing
Data Loading: Download primary datasets, `ESOL` and `Lipophilicity` via MoleculeNet or TDC. Primary dataset used is `ESOL` and confirmation dataset is `Lipophilicity`.

In [ ]:
#dataset
DATASETS = {
    "esol": dict(
        url="https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv",
        smiles="smiles", target="measured log solubility in mols per litre",
        label="ESOL (logS)"),
    "lipo": dict(
        url="https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv",
        smiles="smiles", target="exp", label="Lipophilicity (logD)"),
}

BASE_MODELS = ["XGB-RDKit", "LSTM-SMILES", "GCN"]
STRETCH_MODELS = ["LSTM-SELFIES", "XGB-Mol2Vec"]
#locate pretrained model
MOL2VEC_URL = ("https://raw.githubusercontent.com/samoturk/mol2vec/master/"
               "examples/models/model_300dim.pkl")

#for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

#dataset -> dataframe
def load_raw(name, cache_dir):
    cfg = DATASETS[name]
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{name}.csv")
    if not os.path.exists(path):
        print(f"[data] downloading {name} ...")
        urllib.request.urlretrieve(cfg["url"], path)
    df = pd.read_csv(path)
    return df[[cfg["smiles"], cfg["target"]]].rename(
        columns={cfg["smiles"]: "smiles", cfg["target"]: "y"})

In [ ]:
ESOL = load_raw("esol", cache_dir="data")
ESOL.head()

[data] downloading esol ...


,smiles,y
0,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.77
1,Cc1occc1C(=O)Nc2ccccc2,-3.30
2,CC(C)=CCCC(C)=CC(=O),-2.06
3,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-7.87
4,c1ccsc1,-1.33


In [ ]:
#(cleaning)Curation: parse -> largest fragment (desalt) -> neutralise -> canonical-> merge duplicates (drop those with conflicting labels)

def curate(df, conflict_tol=0.5):
    chooser = rdMolStandardize.LargestFragmentChooser(preferOrganic=True) #remove disconnected components like counterions
    uncharger = rdMolStandardize.Uncharger() #attempt to neutralise formal charge where chemically appropriate
    rows, n_invalid = [], 0  #cleaned molecular structures and target values, rejected entres
    for smi, y in zip(df["smiles"], df["y"]): #y is property
        mol = Chem.MolFromSmiles(str(smi)) #SMILES to RDKit
        if mol is None or pd.isna(y):
            n_invalid += 1
            continue
        try:
            mol = chooser.choose(mol)
            mol = uncharger.uncharge(mol)
            Chem.SanitizeMol(mol) #checks and normalizes impt chemical properties
            rows.append((Chem.MolToSmiles(mol), float(y))) #converts standardized molecule back to SMILES string (canonical SMILES)
        except Exception:
            n_invalid += 1

    d = pd.DataFrame(rows, columns=["smiles", "y"])
    g = d.groupby("smiles")["y"].agg(["mean", "std", "count"]).reset_index()
    n_dup = int((g["count"] > 1).sum()) #count duplicated molecules
    conflict = g["std"].fillna(0) > conflict_tol #identify conflicting target values (std)
    out = g.loc[~conflict, ["smiles", "mean"]].rename(columns={"mean": "y"}) #remove conflicting molecules and keep the mean
    print(f"[curate] raw={len(df)} invalid={n_invalid} duplicated_smiles={n_dup} "
          f"dropped_conflicting={int(conflict.sum())} -> final={len(out)}")
    return out.reset_index(drop=True)

In [ ]:
# 2. Scaffold splits (randomised by seed; identical across all models)

def murcko(smi):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smiles=smi, includeChirality=False) #remove side functional groups
    except Exception:
        return ""


def scaffold_split(scaffolds, seed, frac=(0.8, 0.1, 0.1)): #divide the dataset
    """Randomised scaffold split (Chemprop-style): scaffold sets are never broken
    across splits; very large sets go to train first, remaining sets shuffled."""
    groups = defaultdict(list)
    for i, sc in enumerate(scaffolds):
        groups[sc].append(i)
    sets = list(groups.values())
    n = len(scaffolds)
    n_train, n_val = int(frac[0] * n), int(frac[1] * n) #desired size not separation yet
    n_test = n - n_train - n_val
    big = [s for s in sets if len(s) > min(n_val, n_test) / 2] #groups larger than half the smaller validation/test target
    small = [s for s in sets if len(s) <= min(n_val, n_test) / 2] #groups at or below that threshold
    rng = np.random.RandomState(seed)
    #shuffling the groups, not molecules
    rng.shuffle(big)
    rng.shuffle(small)
    #assign each entire group to a split
    tr, va, te = [], [], [] #train, validate, test
    for s in big + small:
        if len(tr) + len(s) <= n_train:
            tr += s
        elif len(va) + len(s) <= n_val:
            va += s
        else:
            te += s
    return np.array(tr), np.array(va), np.array(te)



In [ ]:
# save curated dataset (esol) as CSV
raw_esol_df = load_raw("esol", cache_dir="data")
df_esol = curate(raw_esol_df)

#extract one scaffold for each molecules
scaffolds =[murcko(smi) for smi in df_esol['smiles']]

os.makedirs("outputs", exist_ok=True)

df_esol.to_csv("outputs/esol_curated.csv", index=False)

print("Saved curated dataset!")
print("Rows:", len(df_esol))
print("Columns:", df_esol.columns.tolist())
print(df_esol.head())


#curation log
import json

curation_log = {
    "dataset": "ESOL",
    "source": DATASETS["esol"]["url"],
    "raw_rows": len(raw_esol_df),
    "invalid_or_missing_rows_removed": 0,
    "duplicate_smiles_groups": 11,
    "conflicting_groups_dropped": 1,
    "final_rows": len(df_esol),
    "standardization": [
        "Parse SMILES using RDKit",
        "Select the largest fragment, preferring organic fragments",
        "Attempt neutralization using RDKit Uncharger",
        "Sanitize molecules",
        "Convert to canonical SMILES"
    ],
    "deduplication": (
        "Group identical canonical SMILES; average target values "
        "when target standard deviation <= 0.5; drop groups above 0.5"
    ),
    "subsampling": "None"
}

with open("outputs/esol_curation_log.json", "w") as f:
    json.dump(curation_log, f, indent=2)

print("Curation log saved!")

[curate] raw=1128 invalid=0 duplicated_smiles=11 dropped_conflicting=1 -> final=1116
Saved curated dataset!
Rows: 1116
Columns: ['smiles', 'y']
                 smiles     y
0         BrC(Br)(Br)Br -3.14
1             BrC(Br)Br -1.91
2                 BrCBr -1.17
3                BrCCBr -1.68
4  Brc1cc(Br)c(Br)cc1Br -6.98
Curation log saved!


In [ ]:
#validating quality
def validate_dataset_and_splits(df, scaffolds, train_idx, val_idx, test_idx):
    """Validate curated data and scaffold splits before model training."""

    n = len(df)
    splits = {
        "Train": np.asarray(train_idx),
        "Val": np.asarray(val_idx),
        "Test": np.asarray(test_idx),
    }

    print("=" * 55)
    print("DATASET AND SCAFFOLD SPLIT VALIDATION")
    print("=" * 55)

    # 1. Basic dataset checks
    assert n > 0, "Dataset is empty."
    assert {"smiles", "y"}.issubset(df.columns), \
        "Dataset must contain 'smiles' and 'y' columns."
    assert df["smiles"].notna().all(), "Missing SMILES found."
    assert df["y"].notna().all(), "Missing targets found."
    assert np.isfinite(df["y"].to_numpy(dtype=float)).all(), \
        "Non-finite target values found."
    assert df["smiles"].is_unique, "Duplicate SMILES found."
    assert len(scaffolds) == n, "Scaffold count does not match dataset."

    print(f"\n[PASS] Dataset contains {n} molecules.")
    print("[PASS] No missing SMILES or target values.")
    print("[PASS] Targets are finite and SMILES are unique.")

    # 2. Check split indices are valid
    for name, idx in splits.items():
        assert idx.ndim == 1, f"{name} indices must be one-dimensional."
        assert np.issubdtype(idx.dtype, np.integer), \
            f"{name} indices must be integers."
        assert len(idx) > 0, f"{name} split is empty."
        assert np.all((idx >= 0) & (idx < n)), \
            f"{name} contains out-of-range indices."
        assert len(np.unique(idx)) == len(idx), \
            f"{name} contains repeated indices."

    # 3. Check molecule assignment: no overlap or omissions
    all_idx = np.concatenate(list(splits.values()))

    assert len(all_idx) == n, \
        "Total split sizes do not equal dataset size."
    assert len(np.unique(all_idx)) == n, \
        "A molecule appears in multiple splits."
    assert set(all_idx.tolist()) == set(range(n)), \
        "Some molecules have not been assigned."

    print("[PASS] Every molecule is assigned exactly once.")

    # 4. Check scaffold overlap
    scaffold_sets = {
        name: {scaffolds[i] for i in idx}
        for name, idx in splits.items()
    }

    names = list(splits)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            overlap = scaffold_sets[a] & scaffold_sets[b]
            assert not overlap, \
                f"Scaffold leakage between {a} and {b}: {len(overlap)} shared."

    print("[PASS] No scaffold overlap between splits.")

    # 5. Report split sizes and proportions
    print("\nSplit sizes:")
    for name, idx in splits.items():
        print(f"{name:5s}: {len(idx):4d} ({len(idx) / n:.1%})")

    # 6. Report target distributions
    print("\nTarget distributions:")
    for name, idx in splits.items():
        y = df.iloc[idx]["y"].astype(float)
        print(
            f"{name:5s}: mean={y.mean():.3f}, "
            f"std={y.std():.3f}, min={y.min():.3f}, "
            f"max={y.max():.3f}"
        )

    # 7. Report empty scaffolds (often acyclic molecules)
    print("\nEmpty-scaffold molecules:")
    for name, idx in splits.items():
        count = sum(scaffolds[i] == "" for i in idx)
        print(f"{name:5s}: {count}")

    print("\nAll validation checks passed!")
    print("=" * 55)

    return True


In [ ]:
#generate and save split manifest

seeds = [42,43,44,45,46]

os.makedirs("outputs", exist_ok=True)

split_manifests = []

for seed in seeds:
    print(f"\n Seed {seed}")
    # Generate the scaffold split for this seed
    train_idx, val_idx, test_idx = scaffold_split(
        scaffolds,
        seed=seed
    )

    # Validate this split using your existing function
    validate_dataset_and_splits(
        df_esol, scaffolds, train_idx, val_idx, test_idx
    )

    # Save molecule identity, original row index, seed and split
    indices = np.concatenate([train_idx, val_idx, test_idx])

    manifest = pd.DataFrame({
        "row_idx": indices,
        "smiles": df_esol.iloc[indices]["smiles"].to_numpy(),
        "seed": seed,
        "split": (
            ["train"] * len(train_idx)
            + ["validation"] * len(val_idx)
            + ["test"] * len(test_idx)
        )
    })

    split_manifests.append(manifest)

# Combine all seeds into one CSV
split_manifest = pd.concat(split_manifests, ignore_index=True)

split_manifest.to_csv(
    "outputs/esol_scaffold_splits.csv",
    index=False
)

print("Saved: outputs/esol_scaffold_splits.csv")
display(split_manifest.head(10))


 Seed 42
DATASET AND SCAFFOLD SPLIT VALIDATION

[PASS] Dataset contains 1116 molecules.
[PASS] No missing SMILES or target values.
[PASS] Targets are finite and SMILES are unique.
[PASS] Every molecule is assigned exactly once.
[PASS] No scaffold overlap between splits.

Split sizes:
Train:  892 (79.9%)
Val  :  111 (9.9%)
Test :  113 (10.1%)

Target distributions:
Train: mean=-2.944, std=2.071, min=-11.600, max=1.580
Val  : mean=-3.704, std=1.951, min=-8.699, max=1.100
Test : mean=-3.277, std=2.282, min=-9.018, max=1.020

Empty-scaffold molecules:
Train: 315
Val  : 0
Test : 0

All validation checks passed!

 Seed 43
DATASET AND SCAFFOLD SPLIT VALIDATION

[PASS] Dataset contains 1116 molecules.
[PASS] No missing SMILES or target values.
[PASS] Targets are finite and SMILES are unique.
[PASS] Every molecule is assigned exactly once.
[PASS] No scaffold overlap between splits.

Split sizes:
Train:  892 (79.9%)
Val  :  111 (9.9%)
Test :  113 (10.1%)

Target distributions:
Train: mean=-3.01

,row_idx,smiles,seed,split
0,4,Brc1cc(Br)c(Br)cc1Br,42,train
1,5,Brc1cc(Br)cc(Br)c1,42,train
2,6,Brc1ccc(Br)c(Br)c1,42,train
3,7,Brc1ccc(Br)cc1,42,train
4,8,Brc1ccc(I)cc1,42,train
5,10,Brc1cccc(Br)c1,42,train
6,12,Brc1ccccc1,42,train
7,13,Brc1ccccc1Br,42,train
8,17,C#CC(C)N(C)C(=O)Nc1ccc(Cl)cc1,42,train
9,18,C#CC(C)OC(=O)Nc1cccc(Cl)c1,42,train


In [ ]:
# Load and verify the saved scaffold assignments
split_manifest = pd.read_csv(
    "outputs/esol_scaffold_splits.csv"
)

# Check that all expected seeds are present
assert set(seeds).issubset(set(split_manifest["seed"]))

# Check that the manifest matches the current dataset
assert split_manifest.groupby("seed")["row_idx"].nunique().eq(len(df_esol)).all()

print("Split manifest loaded successfully.")
print(split_manifest.groupby(["seed", "split"]).size())

Split manifest loaded successfully.
seed  split     
42    test          113
      train         892
      validation    111
43    test          113
      train         892
      validation    111
44    test          113
      train         892
      validation    111
45    test          113
      train         892
      validation    111
46    test          113
      train         892
      validation    111
dtype: int64


##RDKit Representation

In [ ]:
#representations
# RDKit 2D descriptors
def rdkit_descriptors(mols, drop_crippen=False):
    names = [n for n, _ in Descriptors._descList]
    if drop_crippen:  # ablation: MolLogP/MolMR are themselves strong logP/logS predictors
        names = [n for n in names if n not in ("MolLogP", "MolMR")]
    funcs = dict(Descriptors._descList)
    X = np.full((len(mols), len(names)), np.nan, dtype=np.float64)
    #calculate every descriptor
    for i, m in enumerate(mols):
        for j, n in enumerate(names):
            try:
                X[i, j] = funcs[n](m)
            except Exception:
                pass
    # clean invalid values
    X[~np.isfinite(X)] = np.nan
    X[np.abs(X) > 1e8] = np.nan
    return X.astype(np.float32), names

In [ ]:
# generate descriptors for curated ESOL dataset

# Load curated ESOL data
df_curated_esol = pd.read_csv("outputs/esol_curated.csv")

# Convert SMILES to RDKit molecules
mols = [Chem.MolFromSmiles(s) for s in df_curated_esol["smiles"]]

# Safety check
assert all(m is not None for m in mols)

# Calculate descriptors using your existing function
X_desc, descriptor_names = rdkit_descriptors(mols)

y = df_curated_esol["y"].to_numpy(dtype=np.float32)

print("Number of molecules:", len(mols))
print("Descriptor matrix shape:", X_desc.shape)
print("Number of descriptors:", len(descriptor_names))
print("Missing values:", np.isnan(X_desc).sum())

Number of molecules: 1116
Descriptor matrix shape: (1116, 217)
Number of descriptors: 217
Missing values: 5


# Baseline Pipeline and Scaffold Splits
The general workflow used to encode and evaluate the three pipelines (`RDKit with XGBoost`, `SMILES with LSTM` and `Molecular graph with GCN`) is as follows:

- **Feature extraction:** Computation of 2D RDKit chemical descriptors (e.g., Molecular Weight, LogP, TPSA, etc.) for canonicalized molecules
- **Data Scaling:** Normalization of features using StandardScaler
- **Model Training:** Training of XGBoost Regressor on CPU (using default or slightly tuned parameters)
- **Evaluation:** Calculation of RMSE (Root Mean Squared Error) and MAE (Mean Absolute Error) on test splits, save predictions and artifact of the trained model (xgboost_baseline.json).

##RDKit with XGBoost (baseline)

In [ ]:
#XGBoost

xgb_results =[]
xgb_predictions = []

#seed loop

for seed in seeds:
  print(f"\nTraining XGBoost with seed {seed}")

    # Retrieve the EXISTING split for this seed
  seed_manifest = split_manifest[
        split_manifest["seed"] == seed
    ]

  train_idx = seed_manifest.loc[
        seed_manifest["split"] == "train", "row_idx"
    ].to_numpy(dtype=int)

  val_idx = seed_manifest.loc[
        seed_manifest["split"] == "validation", "row_idx"
    ].to_numpy(dtype=int)

  test_idx = seed_manifest.loc[
        seed_manifest["split"] == "test", "row_idx"
    ].to_numpy(dtype=int)

  # Split the descriptors and targets
  X_train = X_desc[train_idx]
  X_val = X_desc[val_idx]
  X_test = X_desc[test_idx]

  y_train = y[train_idx]
  y_val = y[val_idx]
  y_test = y[test_idx]

  # Remove descriptors that are entirely missing in training
  valid_cols = ~np.isnan(X_train).all(axis=0)

  X_train = X_train[:, valid_cols]
  X_val = X_val[:, valid_cols]
  X_test = X_test[:, valid_cols]

  # Fit imputation on training data only
  imputer = SimpleImputer(strategy="median")

  X_train = imputer.fit_transform(X_train)
  X_val = imputer.transform(X_val)
  X_test = imputer.transform(X_test)

  # Confirm there are no invalid values remaining
  assert np.isfinite(X_train).all()
  assert np.isfinite(X_val).all()
  assert np.isfinite(X_test).all()

  print("Training shape:", X_train.shape)
  print("Validation shape:", X_val.shape)
  print("Test shape:", X_test.shape)

  #train baseline XGBoost model
  xgb_model = XGBRegressor(
      n_estimators=500,
      max_depth=5,
      learning_rate=0.03,
      subsample=0.8,
      colsample_bytree=0.8,
      objective="reg:squarederror",
      random_state=seed,
      n_jobs=-1,
      early_stopping_rounds=30
  )

  xgb_model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      verbose=False
  )

  #prediction and test target
  val_pred = xgb_model.predict(X_val)
  test_pred = xgb_model.predict(X_test)


  #record result
  xgb_results.append({
        "seed": seed,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "n_features": X_train.shape[1],
        "best_iteration": xgb_model.best_iteration,

        "val_RMSE": np.sqrt(
            mean_squared_error(y_val, val_pred)
        ),
        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_R2": r2_score(y_val, val_pred),

        "test_RMSE": np.sqrt(
            mean_squared_error(y_test, test_pred)
        ),
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_R2": r2_score(y_test, test_pred)
    })

# per molecule test predictions
  for idx, actual, predicted in zip(
        test_idx, y_test, test_pred
    ):
        xgb_predictions.append({
            "seed": int(seed),
            "row_idx": int(idx),
            "smiles": df_curated_esol.iloc[int(idx)]["smiles"],
            "actual": float(actual),
            "predicted": float(predicted),
            "error": float(predicted - actual),
            "absolute_error": float(abs(predicted - actual))
        })

  latest = xgb_results[-1]

  print(
        f"Test RMSE: {xgb_results[-1]['test_RMSE']:.4f} |"
        f"Test MAE: {xgb_results[-1]['test_MAE']:.4f}| "
        f"Test R2: {xgb_results[-1]['test_R2']:.4f}"
    )

#printing results
results_df = pd.DataFrame(xgb_results)

print("\n=== XGBoost Results Across All Seeds ===")
print(results_df.round(4).to_string(index=False))

metrics = ["test_RMSE", "test_MAE", "test_R2"]

print("\n=== Mean Across Seeds ===")
print(results_df[metrics].mean().round(4))

print("\n=== Standard Deviation Across Seeds ===")
print(results_df[metrics].std(ddof=1).round(4))


# 10. Save metrics and Task 4 predictions
os.makedirs("outputs", exist_ok=True)

results_df.to_csv(
    "outputs/xgb_esol_5_seeds.csv",
    index=False
)

xgb_predictions_df = pd.DataFrame(xgb_predictions)

xgb_predictions_df.to_csv(
    "outputs/xgb_esol_test_predictions.csv",
    index=False
)

display(results_df.round(4))

print("\nSaved:")
print("- outputs/xgb_esol_5_seeds.csv")
print("- outputs/xgb_esol_test_predictions.csv")
print("\nXGBoost training and prediction saving completed.")


Training XGBoost with seed 42
Training shape: (892, 217)
Validation shape: (111, 217)
Test shape: (113, 217)
Test RMSE: 0.8859 |Test MAE: 0.6666| Test R2: 0.8480

Training XGBoost with seed 43
Training shape: (892, 217)
Validation shape: (111, 217)
Test shape: (113, 217)
Test RMSE: 0.7837 |Test MAE: 0.6170| Test R2: 0.8635

Training XGBoost with seed 44
Training shape: (892, 217)
Validation shape: (111, 217)
Test shape: (113, 217)
Test RMSE: 0.6357 |Test MAE: 0.5080| Test R2: 0.9023

Training XGBoost with seed 45
Training shape: (892, 217)
Validation shape: (111, 217)
Test shape: (113, 217)
Test RMSE: 0.8215 |Test MAE: 0.6074| Test R2: 0.8170

Training XGBoost with seed 46
Training shape: (892, 217)
Validation shape: (111, 217)
Test shape: (113, 217)
Test RMSE: 0.9303 |Test MAE: 0.6880| Test R2: 0.7953

=== XGBoost Results Across All Seeds ===
 seed  n_train  n_val  n_test  n_features  best_iteration  val_RMSE  val_MAE  val_R2  test_RMSE  test_MAE  test_R2
   42      892    111     11

,seed,n_train,n_val,n_test,n_features,best_iteration,val_RMSE,val_MAE,val_R2,test_RMSE,test_MAE,test_R2
0,42,892,111,113,217,243,0.7396,0.5719,0.8549,0.8859,0.6666,0.8480
1,43,892,111,113,217,497,0.8146,0.6204,0.8640,0.7837,0.6170,0.8635
2,44,892,111,113,217,399,0.6354,0.4786,0.8946,0.6357,0.5080,0.9023
3,45,892,111,113,217,191,0.8622,0.6296,0.8012,0.8215,0.6074,0.8170
4,46,892,111,113,217,499,0.8836,0.7029,0.8640,0.9303,0.6880,0.7953



Saved:
- outputs/xgb_esol_5_seeds.csv
- outputs/xgb_esol_test_predictions.csv

XGBoost training and prediction saving completed.


##SMILES with LSTM

In [ ]:
#representations
# Token sequences (SMILES chars / SELFIES tokens)
def build_sequences(token_lists):
    vocab = {"<pad>": 0}
    for toks in token_lists:
        for t in toks:
            vocab.setdefault(t, len(vocab))   # token inventory only, no labels -> no leakage
    #finding the longest sequence
    L = max(len(t) for t in token_lists)
    #store the actual sequence lengths
    X = np.zeros((len(token_lists), L), dtype=np.int64)
    lens = np.zeros(len(token_lists), dtype=np.int64)
    for i, toks in enumerate(token_lists):
        X[i, :len(toks)] = [vocab[t] for t in toks]
        lens[i] = len(toks)
    return torch.from_numpy(X), torch.from_numpy(lens), len(vocab)

# Convert character tokens using an existing vocabulary
def encode_with_vocab(token_lists, vocab):
    X = np.zeros(
        (len(token_lists), max(len(t) for t in token_lists)),
        dtype=np.int64
    )
    lens = np.zeros(len(token_lists), dtype=np.int64)

    unk_idx = vocab.get("<unk>", 0) #unknown

    for i, toks in enumerate(token_lists):
        X[i, :len(toks)] = [
            vocab.get(t, unk_idx) for t in toks
        ]
        lens[i] = len(toks)

    return torch.from_numpy(X), torch.from_numpy(lens)


# Build vocabulary using training molecules only
def build_train_vocab(train_tokens):
    vocab = {"<pad>": 0, "<unk>": 1}

    for toks in train_tokens:
        for token in toks:
            vocab.setdefault(token, len(vocab))

    return vocab

In [ ]:

#dataset and batch loader
class SmilesDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = X
        self.lengths = lengths
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]


def collate_smiles(batch):
    X, lengths, y = zip(*batch)

    return (
        nn.utils.rnn.pad_sequence(
            X, batch_first=True, padding_value=0
        ),
        torch.stack(lengths),
        torch.stack(y)
    )

#LSTM model
class LSTMRegressor(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=128,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, X, lengths):
        embedded = self.embedding(X)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        return self.fc(hidden[-1]).squeeze(-1)

#training and evaluation
def evaluate_lstm(model, loader):
    model.eval()
    actual, predicted = [], []

    with torch.no_grad():
        for X, lengths, y_batch in loader:
            X = X.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            pred = model(X, lengths)

            actual.extend(y_batch.cpu().numpy())
            predicted.extend(pred.cpu().numpy())

    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)

    return rmse, mae, actual, predicted


def train_lstm(model, train_loader, val_loader,
               max_epochs=100, patience=12):

    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-3, weight_decay=1e-5
    )
    criterion = nn.MSELoss()

    best_rmse = float("inf")
    best_state = None
    best_epoch = 0
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0
        n_seen = 0

        for X, lengths, y_batch in train_loader:
            X = X.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(X, lengths)
            loss = criterion(pred, y_batch)
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            n_seen += len(y_batch)

        train_mse = total_loss / n_seen
        val_rmse, val_mae, _, _ = evaluate_lstm(
            model, val_loader
        )

        history.append({
            "epoch": epoch,
            "train_mse": train_mse,
            "val_rmse": val_rmse,
            "val_mae": val_mae
        })

        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Train MSE: {train_mse:.4f} | "
                f"Val RMSE: {val_rmse:.4f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

In [ ]:
#train LSTM using saved scaffold splits
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
os.makedirs("outputs/lstm_checkpoints", exist_ok=True)

lstm_results = []
lstm_predictions = []
lstm_histories = {}

for seed in seeds:
    print(f"\n{'=' * 45}")
    print(f"Training LSTM | Seed {seed}")
    print("=" * 45)

    set_seed(seed)

    # Retrieve the EXISTING split for this seed
    seed_manifest = split_manifest[
        split_manifest["seed"] == seed
    ]

    train_idx = seed_manifest.loc[
        seed_manifest["split"] == "train", "row_idx"
    ].to_numpy(dtype=int)

    val_idx = seed_manifest.loc[
        seed_manifest["split"] == "validation", "row_idx"
    ].to_numpy(dtype=int)

    test_idx = seed_manifest.loc[
        seed_manifest["split"] == "test", "row_idx"
    ].to_numpy(dtype=int)

    train_smiles = df_curated_esol.iloc[train_idx]["smiles"].tolist()
    val_smiles = df_curated_esol.iloc[val_idx]["smiles"].tolist()
    test_smiles = df_curated_esol.iloc[test_idx]["smiles"].tolist()

    y_train = df_curated_esol.iloc[train_idx]["y"].to_numpy(dtype=np.float32)
    y_val = df_curated_esol.iloc[val_idx]["y"].to_numpy(dtype=np.float32)
    y_test = df_curated_esol.iloc[test_idx]["y"].to_numpy(dtype=np.float32)

    # Character-level tokenisation
    train_tokens = [list(s) for s in train_smiles]
    val_tokens = [list(s) for s in val_smiles]
    test_tokens = [list(s) for s in test_smiles]

    # Build vocabulary from training SMILES only
    vocab = build_train_vocab(train_tokens)

    # Keep your existing build_sequences() unchanged.
    # Use its logic to encode the training tokens.
    X_train, len_train, _ = build_sequences(train_tokens)

    # Align the vocabulary IDs with build_sequences() output.
    # build_sequences() creates its own vocabulary, so use that
    # vocabulary for all three splits for consistent token IDs.
    vocab_existing = {"<pad>": 0}
    for toks in train_tokens:
        for token in toks:
            vocab_existing.setdefault(token, len(vocab_existing))

    X_train, len_train, _ = build_sequences(train_tokens)

    X_val, len_val = encode_with_vocab(
        val_tokens, vocab_existing
    )
    X_test, len_test = encode_with_vocab(
        test_tokens, vocab_existing
    )

    train_ds = SmilesDataset(X_train, len_train, y_train)
    val_ds = SmilesDataset(X_val, len_val, y_val)
    test_ds = SmilesDataset(X_test, len_test, y_test)

    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_smiles,
        generator=generator
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_smiles
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_smiles
    )

    model = LSTMRegressor(
        vocab_size=len(vocab_existing)
    ).to(DEVICE)

    model, history, best_epoch = train_lstm(
        model, train_loader, val_loader
    )

    lstm_histories[seed] = history

    test_rmse, test_mae, actual, predicted = evaluate_lstm(
        model, test_loader
    )
    test_r2 = r2_score(actual, predicted)

    lstm_results.append({
        "seed": seed,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "best_epoch": best_epoch,
        "test_RMSE": test_rmse,
        "test_MAE": test_mae,
        'test_R2': test_r2,
    })

    lstm_predictions.append(pd.DataFrame({
        "row_idx": test_idx,
        "smiles": test_smiles,
        "true_y": actual,
        "predicted_y": predicted,
        "seed": seed
    }))

    torch.save({
        "model_state_dict": model.state_dict(),
        "vocab": vocab_existing,
        "seed": seed,
        "best_epoch": best_epoch
    }, f"outputs/lstm_checkpoints/lstm_seed_{seed}.pt")

    print(
        f"Best epoch: {best_epoch} | "
        f"Test RMSE: {test_rmse:.4f} | "
        f"Test MAE: {test_mae:.4f} | "
        f"Test R2: {test_r2:.4f}"
    )

lstm_results_df = pd.DataFrame(lstm_results)
lstm_predictions_df = pd.concat(
    lstm_predictions, ignore_index=True
)

lstm_results_df.to_csv(
    "outputs/lstm_esol_5_seeds.csv", index=False
)

lstm_predictions_df.to_csv(
    "outputs/lstm_esol_test_predictions.csv", index=False
)


display(lstm_results_df.round(4))
print("\nSaved:")
print("- outputs/lstm_esol_5_seeds.csv")
print("- outputs/lstm_esol_test_predictions.csv")
print("\nLSTM training and prediction saving completed.")


Training LSTM | Seed 42
Epoch 001 | Train MSE: 7.2859 | Val RMSE: 2.0306
Epoch 010 | Train MSE: 0.6981 | Val RMSE: 0.9468
Epoch 020 | Train MSE: 0.3556 | Val RMSE: 0.9174
Epoch 030 | Train MSE: 0.2537 | Val RMSE: 1.0135
Epoch 040 | Train MSE: 0.1840 | Val RMSE: 0.8846
Early stopping at epoch 47
Best epoch: 35 | Test RMSE: 1.2036 | Test MAE: 0.8916 | Test R2: 0.7194

Training LSTM | Seed 43
Epoch 001 | Train MSE: 7.1899 | Val RMSE: 2.2115
Epoch 010 | Train MSE: 0.6799 | Val RMSE: 1.0137
Epoch 020 | Train MSE: 0.3165 | Val RMSE: 0.9138
Epoch 030 | Train MSE: 0.2360 | Val RMSE: 0.9150
Early stopping at epoch 37
Best epoch: 25 | Test RMSE: 1.0167 | Test MAE: 0.7885 | Test R2: 0.7702

Training LSTM | Seed 44
Epoch 001 | Train MSE: 7.2594 | Val RMSE: 1.9589
Epoch 010 | Train MSE: 0.6491 | Val RMSE: 0.8917
Epoch 020 | Train MSE: 0.4978 | Val RMSE: 0.8318
Epoch 030 | Train MSE: 0.3163 | Val RMSE: 0.7327
Epoch 040 | Train MSE: 0.2106 | Val RMSE: 0.7209
Early stopping at epoch 49
Best epoch: 37

,seed,n_train,n_val,n_test,best_epoch,test_RMSE,test_MAE,test_R2
0,42,892,111,113,35,1.2036,0.8916,0.7194
1,43,892,111,113,25,1.0167,0.7885,0.7702
2,44,892,111,113,37,0.8205,0.6307,0.8372
3,45,892,111,113,25,1.0762,0.7900,0.6859
4,46,892,111,113,15,1.0114,0.7582,0.7581



Saved:
- outputs/lstm_esol_5_seeds.csv
- outputs/lstm_esol_test_predictions.csv

LSTM training and prediction saving completed.


In [ ]:
#printing results

print("\n=== XGBoost Results Across All Seeds ===")
print(lstm_results_df.round(4).to_string(index=False))

metrics = ["test_RMSE", "test_MAE", "test_R2"]

print("\n=== Mean Across Seeds ===")
print(lstm_results_df[metrics].mean().round(4))

print("\n=== Standard Deviation Across Seeds ===")
print(lstm_results_df[metrics].std(ddof=1).round(4))



=== XGBoost Results Across All Seeds ===
 seed  n_train  n_val  n_test  best_epoch  test_RMSE  test_MAE  test_R2
   42      892    111     113          35     1.2036    0.8916   0.7194
   43      892    111     113          25     1.0167    0.7885   0.7702
   44      892    111     113          37     0.8205    0.6307   0.8372
   45      892    111     113          25     1.0762    0.7900   0.6859
   46      892    111     113          15     1.0114    0.7582   0.7581

=== Mean Across Seeds ===
test_RMSE    1.0257
test_MAE     0.7718
test_R2      0.7542
dtype: float64

=== Standard Deviation Across Seeds ===
test_RMSE    0.1384
test_MAE     0.0936
test_R2      0.0571
dtype: float64


##Molecular graph with GCN


In [ ]:
#configuration
SEEDS = [42, 43, 44, 45, 46]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 150
PATIENCE = 20

os.makedirs("outputs", exist_ok=True)

# Confirm that the existing curated dataset is available.
required_vars = ["mols", "y", "split_manifest"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        f"Missing variables: {missing}. Run dataset curation and "
        "scaffold-split creation first."
    )

y = np.asarray(y, dtype=np.float32).reshape(-1)

if len(mols) != len(y):
    raise ValueError(
        f"Dataset mismatch: {len(mols)} molecules but {len(y)} targets."
    )

if any(mol is None for mol in mols):
    raise ValueError("Invalid molecule found in mols. Check curation.")

if not set(SEEDS).issubset(set(split_manifest["seed"].unique())):
    raise ValueError("Split manifest does not contain all five seeds.")

print("Device:", DEVICE)
print("Number of molecules:", len(mols))
print("Seeds:", SEEDS)

Device: cpu
Number of molecules: 1116
Seeds: [42, 43, 44, 45, 46]


In [ ]:
#representations
# Molecular graphs
#atoms are nodes, chemical bonds are edge, atom features describe the node, bond features describe the edges
ATOM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53, 5, 14, 34] #atomic numbers
HYB = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2, Chem.HybridizationType.SP3,
       Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2]


def onehot(v, choices):
    vec = [0.0] * (len(choices) + 1)           # last slot = "other"
    vec[choices.index(v) if v in choices else -1] = 1.0
    return vec


def atom_features(a):
    return (onehot(a.GetAtomicNum(), ATOM_LIST)  #which element the atom is
    + onehot(a.GetTotalDegree(), [0, 1, 2, 3, 4, 5]) #number of neighbouring atoms, counting attached hydrogens
            + onehot(a.GetFormalCharge(), [-1, 0, 1]) #charge assigned to the atom
            + onehot(a.GetTotalNumHs(), [0, 1, 2, 3, 4]) #number of attached hydrogen
            + onehot(a.GetHybridization(), HYB) #hybridization category
            + [float(a.GetIsAromatic()), float(a.IsInRing())]) #whether atom is aromatic, belong to a ring


def mol_to_graph(mol): #rdkit to graph data object
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    src, dst = [], [] #source, destination
    #build the edges
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        src += [i, j]
        dst += [j, i]
    ei = torch.tensor([src, dst], dtype=torch.long) if src else torch.empty((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=ei)

graphs = [mol_to_graph(mol) for mol in mols]

assert len(graphs) == len(y)
assert all(graph.x.shape[0] > 0 for graph in graphs)
assert all(
    graph.edge_index.numel() == 0
    or graph.edge_index.max().item() < graph.x.shape[0]
    for graph in graphs
)

print(f"Graph construction successful: {len(graphs)} molecules.")
print("Node feature dimension:", graphs[0].x.shape[1])


Graph construction successful: 1116 molecules.
Node feature dimension: 38


In [ ]:
#define GCN
class GCN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        num_layers=3,
        dropout=0.1
    ):
        super().__init__()

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        dims = [in_dim] + [hidden_dim] * num_layers

        for i in range(num_layers):
            self.convs.append(
                GCNConv(dims[i], dims[i + 1])
            )
            self.norms.append(
                nn.BatchNorm1d(dims[i + 1])
            )

        self.dropout = dropout

        # Mean pooling + sum pooling gives 2 * hidden_dim features.
        self.head = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )

        # Convert atom representations into one vector per molecule.
        mean_pool = global_mean_pool(x, batch)
        sum_pool = global_add_pool(x, batch)

        molecular_vector = torch.cat(
            [mean_pool, sum_pool], dim=1
        )

        return self.head(molecular_vector).squeeze(-1)


In [ ]:
#prediction and training of GCN model

def graph_step(model, batch):
    batch = batch.to(DEVICE)
    prediction = model(batch)
    target = batch.y.view(-1)

    return prediction, target


@torch.no_grad()
def predict_gcn(model, loader):
    model.eval()
    predictions = []

    for batch in loader:
        batch = batch.to(DEVICE)
        predictions.append(model(batch).cpu().numpy())

    return np.concatenate(predictions)


def fit_gcn(graphs, y, split, seed):
    train_idx, val_idx, test_idx = split

    # Standardise targets using training targets only.
    y_mean = float(y[train_idx].mean())
    y_std = float(y[train_idx].std())

    if not np.isfinite(y_std) or y_std < 1e-8:
        raise ValueError(f"Invalid target standard deviation for seed {seed}.")

    y_scaled = (y - y_mean) / y_std

    def make_loader(indices, shuffle=False):
        samples = []
        for i in indices:
            graph = graphs[int(i)].clone()
            graph.y = torch.tensor(
                [y_scaled[int(i)]], dtype=torch.float
            )
            samples.append(graph)

        return GraphLoader(
            samples, batch_size=BATCH_SIZE, shuffle=shuffle
        )

    train_loader = make_loader(train_idx, shuffle=True)
    val_loader = make_loader(val_idx)
    test_loader = make_loader(test_idx)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = GCN(
        in_dim=graphs[0].x.shape[1],
        hidden_dim=128,
        num_layers=3,
        dropout=0.1
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    best_val_rmse = float("inf")
    best_state = None
    patience_counter = 0
    best_epoch = 0

    for epoch in range(MAX_EPOCHS):
        model.train()

        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()

            prediction = model(batch)
            target = batch.y.view(-1)
            loss = F.mse_loss(prediction, target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=5.0
            )
            optimizer.step()

        # Select checkpoint using validation RMSE, not test results.
        val_pred = predict_gcn(model, val_loader) * y_std + y_mean
        val_rmse = float(
            np.sqrt(mean_squared_error(y[val_idx], val_pred))
        )

        if val_rmse < best_val_rmse - 1e-4:
            best_val_rmse = val_rmse
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f"No valid model checkpoint for seed {seed}.")

    model.load_state_dict(best_state)

    val_pred = predict_gcn(model, val_loader) * y_std + y_mean
    test_pred = predict_gcn(model, test_loader) * y_std + y_mean

    return {
        "val_pred": val_pred,
        "test_pred": test_pred,
        "best_epoch": best_epoch,
        "best_val_rmse": best_val_rmse
    }



In [ ]:
#evaluate identical splits (scaffold)

gcn_results = []
gcn_predictions = []


for seed in seeds:
    print(f"\n{'=' * 45}")
    print(f"Training GCN | Seed {seed}")
    print("=" * 45)


        # Retrieve the EXISTING split for this seed
    seed_manifest = split_manifest[
        split_manifest["seed"] == seed
    ]

    train_idx = seed_manifest.loc[
        seed_manifest["split"] == "train", "row_idx"
    ].to_numpy(dtype=int)

    val_idx = seed_manifest.loc[
        seed_manifest["split"] == "validation", "row_idx"
    ].to_numpy(dtype=int)

    test_idx = seed_manifest.loc[
        seed_manifest["split"] == "test", "row_idx"
    ].to_numpy(dtype=int)

    if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
        raise ValueError(f"An empty split was found for seed {seed}.")

    # Ensure each split is a valid, non-overlapping partition.
    combined = np.concatenate([train_idx, val_idx, test_idx])
    if (
        len(np.unique(combined)) != len(y)
        or set(combined.tolist()) != set(range(len(y)))
    ):
        raise ValueError(
            f"Split manifest for seed {seed} does not partition "
            "the current dataset correctly."
        )

    output = fit_gcn(
        graphs=graphs,
        y=y,
        split=(train_idx, val_idx, test_idx),
        seed=seed
    )

    y_true = y[test_idx]
    y_pred = output["test_pred"]

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    gcn_results.append({
        "seed": seed,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "best_epoch": output["best_epoch"],
        "best_val_RMSE": output["best_val_rmse"],
        "n_train": len(train_idx),
        "n_validation": len(val_idx),
        "n_test": len(test_idx)
    })

    for idx, actual, predicted in zip(test_idx, y_true, y_pred):
        gcn_predictions.append({
            "seed": seed,
            "row_idx": int(idx),
            "smiles": df_curated_esol.iloc[int(idx)]["smiles"],
            "actual": float(actual),
            "predicted": float(predicted),
            "error": float(predicted - actual),
            "absolute_error": float(abs(predicted - actual))
        })

    print(f"Best epoch: {output['best_epoch']}")
    print(f"Test RMSE = {rmse:.4f}")
    print(f"Test MAE  = {mae:.4f}")
    print(f"Test R²   = {r2:.4f}")



Training GCN | Seed 42
Best epoch: 35
Test RMSE = 0.9709
Test MAE  = 0.7315
Test R²   = 0.8174

Training GCN | Seed 43
Best epoch: 36
Test RMSE = 0.9662
Test MAE  = 0.7465
Test R²   = 0.7925

Training GCN | Seed 44
Best epoch: 26
Test RMSE = 0.7928
Test MAE  = 0.6683
Test R²   = 0.8480

Training GCN | Seed 45
Best epoch: 30
Test RMSE = 1.0481
Test MAE  = 0.8010
Test R²   = 0.7021

Training GCN | Seed 46
Best epoch: 38
Test RMSE = 0.8599
Test MAE  = 0.6780
Test R²   = 0.8251


In [ ]:
#summary and save results
results_gcn_df = pd.DataFrame(gcn_results)
predictions_gcn_df = pd.DataFrame(gcn_predictions)

print("\nGCN Test Performance by Seed")
print(results_gcn_df.round(4).to_string(index=False))

summary_gcn_df = results_gcn_df[["RMSE", "MAE", "R2"]].agg(
    ["mean", "std"]
).rename(index={"mean": "Mean", "std": "Standard deviation"})

print("\nMean and Standard Deviation Across Five Seeds")
print(summary_gcn_df.round(4).to_string())

results_gcn_df.to_csv("outputs/gcn_esol_metrics_per_seed.csv", index=False)
summary_gcn_df.to_csv("outputs/gcn_esol_summary.csv")
predictions_gcn_df.to_csv("outputs/gcn_esol_test_predictions.csv", index=False)

print("\nSaved files:")
print("- outputs/gcn_esol_metrics_per_seed.csv")
print("- outputs/gcn_esol_summary.csv")
print("- outputs/gcn_esol_test_predictions.csv")
print("\nGCN pipeline completed successfully.")


GCN Test Performance by Seed
 seed   RMSE    MAE     R2  best_epoch  best_val_RMSE  n_train  n_validation  n_test
   42 0.9709 0.7315 0.8174          35         0.7564      892           111     113
   43 0.9662 0.7465 0.7925          36         0.7937      892           111     113
   44 0.7928 0.6683 0.8480          26         0.7882      892           111     113
   45 1.0481 0.8010 0.7021          30         0.8890      892           111     113
   46 0.8599 0.6780 0.8251          38         0.9938      892           111     113

Mean and Standard Deviation Across Five Seeds
                      RMSE     MAE      R2
Mean                0.9276  0.7251  0.7970
Standard deviation  0.1008  0.0541  0.0567

Saved files:
- outputs/gcn_esol_metrics_per_seed.csv
- outputs/gcn_esol_summary.csv
- outputs/gcn_esol_test_predictions.csv

GCN pipeline completed successfully.


# Molecular-Level Error Analysis
- Residual Analysis: Calculation of absolute errors for each molecule across all three models.

- Structural Correlation: Investigation of which molecules each representation gets wrong: **Do errors correlate with molecular size (heavy atom count) or ring systems or specific heteroatoms (e.g., halogens, nitro groups)?**


In [ ]:
# 1. Load per-molecule predictions from each model
xgb_pred = pd.read_csv("outputs/xgb_esol_test_predictions.csv")
lstm_pred = pd.read_csv("outputs/lstm_esol_test_predictions.csv")
gcn_pred = pd.read_csv("outputs/gcn_esol_test_predictions.csv")

# Standardise prediction column names
lstm_pred = lstm_pred.rename(columns={
    "true_y": "actual",
    "predicted_y": "predicted"
})

# Combine all models
xgb_pred["model"] = "XGBoost"
lstm_pred["model"] = "LSTM"
gcn_pred["model"] = "GCN"

predictions = pd.concat([
    xgb_pred[["seed", "row_idx", "smiles", "actual", "predicted", "model"]],
    lstm_pred[["seed", "row_idx", "smiles", "actual", "predicted", "model"]],
    gcn_pred[["seed", "row_idx", "smiles", "actual", "predicted", "model"]]
], ignore_index=True)

# Calculate prediction errors
predictions["error"] = predictions["predicted"] - predictions["actual"]
predictions["absolute_error"] = predictions["error"].abs()

print("Number of saved test predictions:", len(predictions))
print("\nMean absolute error by model:")
print(predictions.groupby("model")["absolute_error"].mean().round(4))


# 2. Extract molecular properties from SMILES
def get_molecular_features(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return {
            "MolWt": np.nan,
            "HeavyAtoms": np.nan,
            "Rings": np.nan,
            "Heteroatoms": np.nan
        }

    return {
        "MolWt": Descriptors.MolWt(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "Heteroatoms": Lipinski.NumHeteroatoms(mol)
    }

# Calculate features once per unique molecule
unique_molecules = predictions[
    ["row_idx", "smiles"]
].drop_duplicates("row_idx").copy()

feature_values = pd.DataFrame(
    unique_molecules["smiles"].apply(get_molecular_features).tolist(),
    index=unique_molecules.index
)

unique_molecules = pd.concat(
    [unique_molecules, feature_values], axis=1
)

predictions = predictions.merge(
    unique_molecules.drop(columns="smiles"),
    on="row_idx",
    how="left",
    validate="many_to_one"
)


# 3. Correlation between molecular features and absolute error
features = ["MolWt", "HeavyAtoms", "Rings", "Heteroatoms"]

correlation_rows = []

for model_name, group in predictions.groupby("model"):
    for feature in features:
        valid = group[[feature, "absolute_error"]].dropna()

        if len(valid) >= 3 and valid[feature].nunique() > 1:
            rho, p_value = spearmanr(
                valid[feature], valid["absolute_error"]
            )
        else:
            rho, p_value = np.nan, np.nan

        correlation_rows.append({
            "model": model_name,
            "feature": feature,
            "spearman_rho": rho,
            "p_value": p_value,
            "n": len(valid)
        })

correlation_df = pd.DataFrame(correlation_rows)

print("\n=== Molecular Features vs Absolute Error ===")
print(correlation_df.round(4).to_string(index=False))

correlation_df.to_csv(
    "outputs/error_correlations.csv", index=False
)


# 4. Compare average errors across molecular-size groups
predictions["SizeGroup"] = pd.qcut(
    predictions["HeavyAtoms"],
    q=3,
    labels=["Small", "Medium", "Large"],
    duplicates="drop"
)

size_results = predictions.groupby(
    ["model", "SizeGroup"], observed=True
).agg(
    MAE=("absolute_error", "mean"),
    MoleculePredictions=("absolute_error", "size")
).reset_index()

print("\n=== Error by Molecular Size ===")
print(size_results.round(4).to_string(index=False))

size_results.to_csv(
    "outputs/error_by_molecular_size.csv", index=False
)


# 5. Identify the most difficult molecules for each model
hardest = (
    predictions.sort_values("absolute_error", ascending=False)
    .groupby("model", group_keys=False)
    .head(10)
    [["model", "seed", "row_idx", "smiles",
      "actual", "predicted", "absolute_error",
      "MolWt", "HeavyAtoms", "Rings", "Heteroatoms"]]
)

print("\n=== Ten Largest Errors per Model ===")
print(hardest.round(4).to_string(index=False))

hardest.to_csv(
    "outputs/hardest_molecules.csv", index=False
)

print("\nTask 4 analysis completed.")

Number of saved test predictions: 1695

Mean absolute error by model:
model
GCN        0.7251
LSTM       0.7718
XGBoost    0.6174
Name: absolute_error, dtype: float64

=== Molecular Features vs Absolute Error ===
  model     feature  spearman_rho  p_value   n
    GCN       MolWt       -0.1372   0.0011 565
    GCN  HeavyAtoms       -0.1056   0.0120 565
    GCN       Rings       -0.0688   0.1024 565
    GCN Heteroatoms       -0.0763   0.0700 565
   LSTM       MolWt       -0.0366   0.3846 565
   LSTM  HeavyAtoms       -0.0197   0.6398 565
   LSTM       Rings        0.0455   0.2806 565
   LSTM Heteroatoms       -0.0243   0.5639 565
XGBoost       MolWt       -0.0913   0.0300 565
XGBoost  HeavyAtoms       -0.0477   0.2580 565
XGBoost       Rings        0.0388   0.3578 565
XGBoost Heteroatoms       -0.0817   0.0522 565

=== Error by Molecular Size ===
  model SizeGroup    MAE  MoleculePredictions
    GCN     Small 0.8405                  199
    GCN    Medium 0.5799                  210
    G

In [ ]:
#identify the molecules each model gets wrong
# Load predictions from the three models

files = {
    "LSTM": "outputs/lstm_esol_test_predictions.csv",
    "XGBoost": "outputs/xgb_esol_test_predictions.csv",
    "GCN": "outputs/gcn_esol_test_predictions.csv"
}

frames = []

for model, filename in files.items():
    df = pd.read_csv(filename)

    # Standardise column names across the three files
    df = df.rename(columns={
        "true_y": "actual",
        "predicted_y": "predicted"
    })

    required = {"seed", "row_idx", "actual", "predicted"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{model} file is missing columns: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df["model"] = model
    df["actual"] = df["actual"].round(4)
    df["error"] = df["predicted"] - df["actual"]
    df["absolute_error"] = df["error"].abs()
    frames.append(df)

predictions = pd.concat(frames, ignore_index=True)

# Check that the same molecule and seed have consistent actual values
check = predictions.groupby(["seed", "row_idx"])["actual"].nunique()
if (check > 1).any():
    raise ValueError(
        "Actual values disagree across models for some seed/molecule pairs."
    )


# A. Molecules with the largest errors for each model

top_errors = (
    predictions.sort_values("absolute_error", ascending=False)
    .groupby("model", group_keys=False)
    .head(10)
    .sort_values(["model", "absolute_error"], ascending=[True, False])
)

print("\nTOP 10 LARGEST ERRORS FOR EACH MODEL")
print(top_errors[
    ["model", "seed", "row_idx", "actual", "predicted",
     "error", "absolute_error"]
].to_string(index=False))


# B. Molecules that are repeatedly difficult across seeds

# Aggregate by molecule, since a molecule can appear in
# the test set in more than one seed.
molecule_errors = (
    predictions.groupby(["model", "row_idx"])
    .agg(
        mean_absolute_error=("absolute_error", "mean"),
        mean_error=("error", "mean"),
        times_tested=("seed", "nunique"),
        actual=("actual", "mean")
    )
    .reset_index()
)

print("\nTOP 10 MOLECULES BY MEAN ABSOLUTE ERROR")
for model in files:
    subset = (
        molecule_errors[molecule_errors["model"] == model]
        .sort_values("mean_absolute_error", ascending=False)
        .head(10)
    )
    print(f"\n{model}")
    print(subset.to_string(index=False))


# C. Compare models on the exact same molecule and seed

wide = predictions.pivot(
    index=["seed", "row_idx", "actual"],
    columns="model",
    values="absolute_error"
).reset_index()

error_cols = ["LSTM", "XGBoost", "GCN"]
wide["models_available"] = wide[error_cols].notna().sum(axis=1)

# Only retain molecule-seed cases evaluated by all 3 models
paired = wide[wide["models_available"] == 3].copy()

# Model with the lowest error for each shared test case
paired["best_model"] = paired[error_cols].idxmin(axis=1)

print("\nBEST MODEL FOR THE SAME MOLECULE-SEED CASES")
print(paired["best_model"].value_counts())

# How often do all three models struggle on the same case?
paired["all_models_large_error"] = (
    paired[error_cols] > 1.0
).all(axis=1)

print(
    "\nShared cases where all 3 models have absolute error > 1.0:",
    int(paired["all_models_large_error"].sum()),
    "out of", len(paired)
)

# Save tables for your report
Path("outputs").mkdir(exist_ok=True)
top_errors.to_csv("outputs/top_10_errors_per_model.csv", index=False)
molecule_errors.to_csv("outputs/mean_error_by_molecule.csv", index=False)
paired.to_csv("outputs/paired_model_comparison.csv", index=False)

print("\nSaved the analysis tables in the outputs folder.")


TOP 10 LARGEST ERRORS FOR EACH MODEL
  model  seed  row_idx  actual  predicted     error  absolute_error
    GCN    42      956  -3.931  -7.741331 -3.810331        3.810331
    GCN    45      247  -7.320  -3.603565  3.716435        3.716435
    GCN    45      998  -4.445  -7.724419 -3.279419        3.279419
    GCN    45      933  -5.259  -8.518580 -3.259580        3.259580
    GCN    46      676   0.715  -2.344192 -3.059192        3.059192
    GCN    45      956  -3.931  -6.802479 -2.871479        2.871479
    GCN    42      676   0.715  -2.122845 -2.837845        2.837845
    GCN    46      528  -8.600  -6.031962  2.568038        2.568038
    GCN    45      843  -0.170  -2.604848 -2.434848        2.434848
    GCN    43      910  -3.401  -1.011839  2.389161        2.389161
   LSTM    42      956  -3.931  -8.783830 -4.852830        4.852830
   LSTM    45      998  -4.445  -8.310625 -3.865625        3.865625
   LSTM    46      676   0.715  -3.061017 -3.776017        3.776017
   LSTM   

In [ ]:
#add molecular features to the error table
# Get one SMILES string per molecule from the prediction files
smiles_map = (
    predictions.dropna(subset=["smiles"])
    .drop_duplicates("row_idx")
    .set_index("row_idx")["smiles"]
    .to_dict()
)

def get_features(smiles):
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None

    if mol is None:
        return pd.Series({
            "MolWt": np.nan,
            "HeavyAtoms": np.nan,
            "Rings": np.nan,
            "Heteroatoms": np.nan
        })

    return pd.Series({
        "MolWt": Descriptors.MolWt(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "Rings": Lipinski.RingCount(mol),
        "Heteroatoms": sum(
            atom.GetAtomicNum() not in (1, 6)
            for atom in mol.GetAtoms()
        )
    })

feature_table = pd.DataFrame.from_dict(
    smiles_map, orient="index"
).rename(columns={0: "smiles"})
feature_table.index.name = "row_idx"
feature_table = feature_table.join(
    feature_table["smiles"].apply(get_features)
)

# Attach the molecular features to each model's error table
detailed_errors = molecule_errors.merge(
    feature_table.reset_index(),
    on="row_idx",
    how="left"
)

# Display the most consistently difficult molecules per model
for model in ["LSTM", "XGBoost", "GCN"]:
    print(f"\n{model}: hardest molecules")
    print(
        detailed_errors[detailed_errors["model"] == model]
        .sort_values("mean_absolute_error", ascending=False)
        .head(10)[[
            "row_idx", "smiles", "actual",
            "mean_absolute_error", "times_tested",
            "MolWt", "HeavyAtoms", "Rings", "Heteroatoms"
        ]].to_string(index=False)
    )

detailed_errors.to_csv(
    "outputs/molecule_errors_with_features.csv",
    index=False
)


LSTM: hardest molecules
 row_idx                                                                smiles  actual  mean_absolute_error  times_tested   MolWt  HeavyAtoms  Rings  Heteroatoms
     956 O=C1NC(=O)C2C3C(C(O)(c4ccccc4)c4ccccn4)=CC(C3=C(c3ccccc3)c3ccccn3)C12  -3.931             4.072993             2 511.581        39.0    7.0          6.0
     676                                              Cc1cc(=O)n(-c2ccccc2)n1C   0.715             3.692197             2 188.230        14.0    2.0          3.0
     665                                       Cc1c(N(C)C)c(=O)n(-c2ccccc2)n1C  -0.364             3.029697             2 231.299        17.0    2.0          4.0
     999                                     O=c1oc2ccccc2c(O)c1C1CCCc2ccccc21  -2.840             2.842432             1 292.334        22.0    4.0          3.0
     998            O=c1oc2ccccc2c(O)c1C(CC(O)c1ccc(-c2ccc(Br)cc2)cc1)c1ccccc1  -4.445             2.791055             2 527.414        35.0    5.0          5.0
   

# SELFIES-Based Model

In [ ]:
BATCH_SIZE = 32
os.makedirs("outputs/selfies_checkpoints", exist_ok=True)

def smiles_to_selfies_tokens(smiles):
    """Convert a SMILES string to a list of SELFIES tokens."""
    selfies_string = sf.encoder(smiles)
    if selfies_string is None:
        raise ValueError(f"SELFIES conversion failed for: {smiles}")
    return list(sf.split_selfies(selfies_string))


selfies_results = []
selfies_predictions = []

for seed in seeds:
    print(f"\n{'=' * 45}")
    print(f"Training SELFIES-LSTM | Seed {seed}")
    print("=" * 45)

    set_seed(seed)

    seed_manifest = split_manifest[
        split_manifest["seed"] == seed
    ]

    train_idx = seed_manifest.loc[
        seed_manifest["split"] == "train", "row_idx"
    ].to_numpy(dtype=int)

    val_idx = seed_manifest.loc[
        seed_manifest["split"] == "validation", "row_idx"
    ].to_numpy(dtype=int)

    test_idx = seed_manifest.loc[
        seed_manifest["split"] == "test", "row_idx"
    ].to_numpy(dtype=int)

    train_smiles = df_curated_esol.iloc[train_idx]["smiles"].tolist()
    val_smiles = df_curated_esol.iloc[val_idx]["smiles"].tolist()
    test_smiles = df_curated_esol.iloc[test_idx]["smiles"].tolist()

    y_train = df_curated_esol.iloc[train_idx]["y"].to_numpy(dtype=np.float32)
    y_val = df_curated_esol.iloc[val_idx]["y"].to_numpy(dtype=np.float32)
    y_test = df_curated_esol.iloc[test_idx]["y"].to_numpy(dtype=np.float32)

    # Tokenise each split separately.
    train_tokens = [smiles_to_selfies_tokens(s) for s in train_smiles]
    val_tokens = [smiles_to_selfies_tokens(s) for s in val_smiles]
    test_tokens = [smiles_to_selfies_tokens(s) for s in test_smiles]

    # Learn vocabulary from training molecules only.
    vocab = build_train_vocab(train_tokens)

    X_train, len_train = encode_with_vocab(train_tokens, vocab)
    X_val, len_val = encode_with_vocab(val_tokens, vocab)
    X_test, len_test = encode_with_vocab(test_tokens, vocab)

    train_ds = SmilesDataset(X_train, len_train, y_train)
    val_ds = SmilesDataset(X_val, len_val, y_val)
    test_ds = SmilesDataset(X_test, len_test, y_test)

    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_smiles,
        generator=generator
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, collate_fn=collate_smiles
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE,
        shuffle=False, collate_fn=collate_smiles
    )

    model = LSTMRegressor(
        vocab_size=len(vocab)
    ).to(DEVICE)

    model, history, best_epoch = train_lstm(
        model, train_loader, val_loader,
        max_epochs=100, patience=12
    )

    test_rmse, test_mae, actual, predicted = evaluate_lstm(
        model, test_loader
    )
    test_r2 = r2_score(actual, predicted)

    selfies_results.append({
        "seed": seed,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "best_epoch": best_epoch,
        "RMSE": test_rmse,
        "MAE": test_mae,
        "R2": test_r2
    })

    selfies_predictions.append(pd.DataFrame({
        "row_idx": test_idx,
        "smiles": test_smiles,
        "actual": actual,
        "predicted": predicted,
        "seed": seed
    }))

    torch.save({
        "model_state_dict": model.state_dict(),
        "vocab": vocab,
        "seed": seed,
        "best_epoch": best_epoch
    }, f"outputs/selfies_checkpoints/selfies_seed_{seed}.pt")

    print(
        f"Best epoch: {best_epoch} | "
        f"RMSE: {test_rmse:.4f} | "
        f"MAE: {test_mae:.4f} | R2: {test_r2:.4f}"
    )

selfies_results_df = pd.DataFrame(selfies_results)
selfies_predictions_df = pd.concat(
    selfies_predictions, ignore_index=True
)

selfies_results_df.to_csv(
    "outputs/selfies_esol_metrics_per_seed.csv", index=False
)
selfies_predictions_df.to_csv(
    "outputs/selfies_esol_test_predictions.csv", index=False
)

print("\nSELFIES-LSTM results:")
display(selfies_results_df.round(4))

print("\nSaved files:")
print("outputs/selfies_esol_metrics_per_seed.csv")
print("outputs/selfies_esol_test_predictions.csv")




Training SELFIES-LSTM | Seed 42
Epoch 001 | Train MSE: 7.0222 | Val RMSE: 2.0376
Epoch 010 | Train MSE: 0.6182 | Val RMSE: 0.9432
Epoch 020 | Train MSE: 0.2908 | Val RMSE: 0.9406
Early stopping at epoch 23
Best epoch: 11 | RMSE: 1.1574 | MAE: 0.9028 | R2: 0.7406

Training SELFIES-LSTM | Seed 43
Epoch 001 | Train MSE: 6.9517 | Val RMSE: 2.2794
Epoch 010 | Train MSE: 0.7455 | Val RMSE: 1.1494
Epoch 020 | Train MSE: 0.3845 | Val RMSE: 1.0812
Epoch 030 | Train MSE: 0.2461 | Val RMSE: 1.0411
Epoch 040 | Train MSE: 0.1653 | Val RMSE: 1.0424
Epoch 050 | Train MSE: 0.1482 | Val RMSE: 1.0261
Epoch 060 | Train MSE: 0.1261 | Val RMSE: 1.0195
Epoch 070 | Train MSE: 0.1180 | Val RMSE: 1.0282
Early stopping at epoch 78
Best epoch: 66 | RMSE: 1.0731 | MAE: 0.8508 | R2: 0.7440

Training SELFIES-LSTM | Seed 44
Epoch 001 | Train MSE: 7.6172 | Val RMSE: 1.9565
Epoch 010 | Train MSE: 0.7025 | Val RMSE: 1.0724
Epoch 020 | Train MSE: 0.3584 | Val RMSE: 0.8963
Epoch 030 | Train MSE: 0.2233 | Val RMSE: 0.911

,seed,n_train,n_val,n_test,best_epoch,RMSE,MAE,R2
0,42,892,111,113,11,1.1574,0.9028,0.7406
1,43,892,111,113,66,1.0731,0.8508,0.7440
2,44,892,111,113,20,0.8203,0.6269,0.8373
3,45,892,111,113,17,0.9981,0.7667,0.7298
4,46,892,111,113,18,0.9857,0.7728,0.7702



Saved files:
outputs/selfies_esol_metrics_per_seed.csv
outputs/selfies_esol_test_predictions.csv


# Mol2Vec-Based Model

In [ ]:
MOL2VEC_URL = (
    "https://raw.githubusercontent.com/samoturk/mol2vec/"
    "master/examples/models/model_300dim.pkl"
)
MOL2VEC_PATH = "outputs/model_300dim.pkl"
os.makedirs("outputs", exist_ok=True)

if not os.path.exists(MOL2VEC_PATH):
    urllib.request.urlretrieve(MOL2VEC_URL, MOL2VEC_PATH)

# Load pretrained vectors.
mol2vec_model = Word2Vec.load(MOL2VEC_PATH)
wv = mol2vec_model.wv
embedding_dim = wv.vector_size

print("Pretrained Mol2Vec embedding dimension:", embedding_dim)


def mol2vec_sentence(mol, radius=1):
    """
    Generate Morgan substructure tokens indexed by atom and radius.
    """
    bit_info = {}

    # Correct API: generate the fingerprint and populate bit_info.
    AllChem.GetMorganFingerprint(
        mol,
        radius=radius,
        bitInfo=bit_info
    )

    per_atom = [
        {r: None for r in range(radius + 1)}
        for _ in range(mol.GetNumAtoms())
    ]

    for bit_id, environments in bit_info.items():
        for atom_idx, rad in environments:
            per_atom[atom_idx][rad] = str(bit_id)

    return [
        per_atom[atom_idx][rad]
        for atom_idx in range(mol.GetNumAtoms())
        for rad in range(radius + 1)
        if per_atom[atom_idx][rad] is not None
    ]

def mol2vec_features(mols, model_wv):
    """
    Sum pretrained substructure vectors into one vector per molecule.
    Unknown tokens use the pretrained UNK vector if available.
    """
    X = np.zeros(
        (len(mols), model_wv.vector_size), dtype=np.float32
    )
    unk_vector = model_wv["UNK"] if "UNK" in model_wv else None

    for i, mol in enumerate(mols):
        tokens = mol2vec_sentence(mol)

        for token in tokens:
            if token in model_wv:
                X[i] += model_wv[token]
            elif unk_vector is not None:
                X[i] += unk_vector
            # Otherwise, skip tokens absent from the pretrained vocabulary.

    return X


# Ensure this is the same curated dataframe/order used by split_manifest.
df = pd.read_csv("outputs/esol_curated.csv")
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]

if any(mol is None for mol in mols):
    raise ValueError("Invalid molecule found in the curated dataset.")

X_mol2vec = mol2vec_features(mols, wv)
y = df["y"].to_numpy(dtype=np.float32)

print("Mol2Vec feature matrix:", X_mol2vec.shape)

mol2vec_results = []
mol2vec_predictions = []

for seed in seeds:
    print(f"\nTraining Mol2Vec-XGBoost | Seed {seed}")

    seed_manifest = split_manifest[
        split_manifest["seed"] == seed
    ]

    train_idx = seed_manifest.loc[
        seed_manifest["split"] == "train", "row_idx"
    ].to_numpy(dtype=int)
    val_idx = seed_manifest.loc[
        seed_manifest["split"] == "validation", "row_idx"
    ].to_numpy(dtype=int)
    test_idx = seed_manifest.loc[
        seed_manifest["split"] == "test", "row_idx"
    ].to_numpy(dtype=int)

    X_train, X_val, X_test = (
        X_mol2vec[train_idx],
        X_mol2vec[val_idx],
        X_mol2vec[test_idx]
    )
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    model = XGBRegressor(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=-1
    )

    # Validation data is used for early stopping where supported.
    try:
        model.set_params(early_stopping_rounds=40)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    except TypeError:
        # Compatibility fallback for older XGBoost versions.
        model = XGBRegressor(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=-1
        )
        model.fit(X_train, y_train, verbose=False)

    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    mol2vec_results.append({
        "seed": seed,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    mol2vec_predictions.append(pd.DataFrame({
        "row_idx": test_idx,
        "smiles": df.iloc[test_idx]["smiles"].to_numpy(),
        "actual": y_test,
        "predicted": pred,
        "seed": seed
    }))

    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

mol2vec_results_df = pd.DataFrame(mol2vec_results)
mol2vec_predictions_df = pd.concat(
    mol2vec_predictions, ignore_index=True
)

mol2vec_results_df.to_csv(
    "outputs/mol2vec_esol_metrics_per_seed.csv", index=False
)
mol2vec_predictions_df.to_csv(
    "outputs/mol2vec_esol_test_predictions.csv", index=False
)

print("\nMol2Vec-XGBoost results:")
display(mol2vec_results_df.round(4))


Pretrained Mol2Vec embedding dimension: 300
Mol2Vec feature matrix: (1116, 300)

Training Mol2Vec-XGBoost | Seed 42
RMSE: 1.0981 | MAE: 0.8534 | R2: 0.7665

Training Mol2Vec-XGBoost | Seed 43
RMSE: 1.2009 | MAE: 0.9339 | R2: 0.6794

Training Mol2Vec-XGBoost | Seed 44
RMSE: 0.9999 | MAE: 0.8069 | R2: 0.7583

Training Mol2Vec-XGBoost | Seed 45
RMSE: 1.0407 | MAE: 0.8375 | R2: 0.7063

Training Mol2Vec-XGBoost | Seed 46
RMSE: 1.1104 | MAE: 0.8371 | R2: 0.7084

Mol2Vec-XGBoost results:


,seed,n_train,n_val,n_test,RMSE,MAE,R2
0,42,892,111,113,1.0981,0.8534,0.7665
1,43,892,111,113,1.2009,0.9339,0.6794
2,44,892,111,113,0.9999,0.8069,0.7583
3,45,892,111,113,1.0407,0.8375,0.7063
4,46,892,111,113,1.1104,0.8371,0.7084


#Simple Test Ensemble

In [ ]:
prediction_files = {
    "XGB-RDKit": "outputs/xgb_esol_test_predictions.csv",
    "LSTM-SMILES": "outputs/lstm_esol_test_predictions.csv",
    "GCN": "outputs/gcn_esol_test_predictions.csv",
    "LSTM-SELFIES": "outputs/selfies_esol_test_predictions.csv",
    "XGB-Mol2Vec": "outputs/mol2vec_esol_test_predictions.csv",
}

def load_predictions(model_name, path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing predictions for {model_name}: {path}. "
            "Run that model's training cell first."
        )

    pred = pd.read_csv(path)

    # Standardise column names from the existing pipelines.
    aliases = {
        "true_y": "actual",
        "y_true": "actual",
        "predicted_y": "predicted",
        "y_pred": "predicted",
        "test_pred": "predicted",
    }
    pred = pred.rename(columns=aliases)

    required = {"seed", "row_idx", "actual", "predicted"}
    missing = required - set(pred.columns)
    if missing:
        raise ValueError(
            f"{model_name} prediction file is missing columns: {missing}. "
            f"Found: {pred.columns.tolist()}"
        )

    pred = pred[["seed", "row_idx", "actual", "predicted"]].copy()
    pred["seed"] = pred["seed"].astype(int)
    pred["row_idx"] = pred["row_idx"].astype(int)

    if pred.duplicated(["seed", "row_idx"]).any():
        raise ValueError(
            f"{model_name} has duplicate seed/row_idx predictions."
        )

    return pred.rename(columns={
        "actual": f"actual_{model_name}",
        "predicted": f"pred_{model_name}"
    })


# Keep only molecule-seed cases predicted by every model.
merged = None

for model_name, path in prediction_files.items():
    pred = load_predictions(model_name, path)

    if merged is None:
        merged = pred
    else:
        merged = merged.merge(
            pred, on=["seed", "row_idx"], how="inner",
            validate="one_to_one"
        )

model_names = list(prediction_files.keys())

# Confirm all models agree on the target values.
actual_columns = [f"actual_{m}" for m in model_names]
actual_matrix = merged[actual_columns].to_numpy()

if not np.allclose(actual_matrix, actual_matrix[:, [0]], atol=1e-5):
    raise ValueError(
        "Actual targets differ between model files. "
        "Check that all predictions use the same curated dataset."
    )

merged["actual"] = actual_matrix[:, 0]

# Equal-weight mean ensemble.
prediction_columns = [f"pred_{m}" for m in model_names]
merged["pred_Ensemble-Mean"] = merged[prediction_columns].mean(axis=1)

print("Common molecule-seed cases:", len(merged))
print("Seeds included:", sorted(merged["seed"].unique()))

# Compare all models on exactly the same cases.
all_models = model_names + ["Ensemble-Mean"]
metric_rows = []

for model_name in all_models:
    pred_col = (
        "pred_Ensemble-Mean"
        if model_name == "Ensemble-Mean"
        else f"pred_{model_name}"
    )

    actual = merged["actual"].to_numpy()
    predicted = merged[pred_col].to_numpy()

    metric_rows.append({
        "Model": model_name,
        "N": len(merged),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
        "R2": r2_score(actual, predicted)
    })

ensemble_comparison = pd.DataFrame(metric_rows).sort_values("MAE")

# Also report metrics for each seed.
seed_rows = []
for seed, group in merged.groupby("seed"):
    actual = group["actual"].to_numpy()

    for model_name in all_models:
        pred_col = (
            "pred_Ensemble-Mean"
            if model_name == "Ensemble-Mean"
            else f"pred_{model_name}"
        )
        predicted = group[pred_col].to_numpy()

        seed_rows.append({
            "seed": seed,
            "Model": model_name,
            "N": len(group),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "MAE": mean_absolute_error(actual, predicted),
            "R2": r2_score(actual, predicted)
        })

ensemble_by_seed = pd.DataFrame(seed_rows)

# Save results and molecule-level ensemble predictions.
merged.to_csv("outputs/esol_five_model_ensemble_predictions.csv", index=False)
ensemble_comparison.to_csv(
    "outputs/esol_five_model_ensemble_comparison.csv", index=False
)
ensemble_by_seed.to_csv(
    "outputs/esol_five_model_ensemble_by_seed.csv", index=False
)

print("\n=== Five-model comparison on common test cases ===")
display(ensemble_comparison.round(4))

print("\n=== Metrics by seed ===")
display(ensemble_by_seed.round(4))

Common molecule-seed cases: 565
Seeds included: [np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]

=== Five-model comparison on common test cases ===


,Model,N,RMSE,MAE,R2
0,XGB-RDKit,565,0.8177,0.6174,0.8476
5,Ensemble-Mean,565,0.8570,0.6523,0.8326
2,GCN,565,0.9320,0.7251,0.8020
1,LSTM-SMILES,565,1.0331,0.7718,0.7567
3,LSTM-SELFIES,565,1.0131,0.7840,0.7660
4,XGB-Mol2Vec,565,1.0921,0.8537,0.7281



=== Metrics by seed ===


,seed,Model,N,RMSE,MAE,R2
0,42,XGB-RDKit,113,0.8859,0.6666,0.8480
1,42,LSTM-SMILES,113,1.2036,0.8916,0.7194
2,42,GCN,113,0.9709,0.7315,0.8174
3,42,LSTM-SELFIES,113,1.1574,0.9028,0.7406
4,42,XGB-Mol2Vec,113,1.0981,0.8534,0.7665
5,42,Ensemble-Mean,113,0.9643,0.7293,0.8199
6,43,XGB-RDKit,113,0.7837,0.6170,0.8635
7,43,LSTM-SMILES,113,1.0167,0.7885,0.7702
8,43,GCN,113,0.9662,0.7465,0.7925
9,43,LSTM-SELFIES,113,1.0731,0.8508,0.7440


###Happy Ending?